<h1 align="center">SWDB Problem Set: Becoming a Data Detective</h1>
<h3 align="center">From someone else's figure to your own analysis</h3>
<p align="center"><i>Works with any SWDB dataset &mdash; bring the one your chosen figure came from.</i></p>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>How this problem set works</h2>

This morning you explored a dataset and made figures. Those figures are now posted on Slack.

**Your starting point is one of your classmates' figures.** Pick any figure from the channel, along
with the dataset it came from &mdash; ideally one you did *not* work on this morning.

| Part | Task |
| --- | --- |
| 1 | Load their dataset and find the pieces the figure needs |
| 2 | Reproduce the figure, and interrogate what it shows |
| 3 | Align activity to event onsets: raster and PSTH |
| 4 | Signal and noise correlations, and whether to trust them |

You already have the data-access skills for Part 1 from this morning's tutorial. This problem set is
about what comes after loading: **shaping data, and checking whether the result means anything.**

**Deliverable:** a short README naming the figure and dataset you chose, the decisions you made at
each step, and an honest assessment of what your numbers do and do not support.

<b>Every dataset is different, and the notebook does not know which one you picked.</b> The code
cells are prompts, not templates &mdash; you write what goes in them, using the access patterns from
this morning. Only a few things are given: the imports, and two helper functions from the tutorial.

The differences you will run into are not cosmetic. Across the datasets in this workshop:

- **Recording modality** &mdash; a continuous calcium signal in some, discrete spike times in
  others. Spikes need binning before anything here applies.
- **Sampling rate** &mdash; from a few Hz to tens of kHz, which sets what timing you can resolve.
- **Number of neurons** &mdash; tens to thousands, which changes what is tractable in one pass.
- **Stimulus structure** &mdash; many conditions with few repeats, few conditions with many, or no
  sensory stimulus at all.
- **What was recorded alongside** &mdash; running, licking, pupil, reward; some datasets have all of
  it, some none.
- **Where things live in the file** &mdash; container and column names differ, and so does which
  container holds the trial table.

None of that is written on the outside of the file. **You have to look.** Part of each prompt is
deciding whether the analysis it asks for even applies to your dataset &mdash; and saying so when it
does not.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h2>Taking it slow: Analysis step by step</h2>

You can now generate an analysis faster than you can check one. Ask an LLM for a correlation matrix
and you will have one in thirty seconds, beautifully formatted, with a colorbar.

The problem is that a result computed on four trials can look exactly like a result computed on four
hundred. A bug can look exactly like a finding. A correlation computed in a window where nothing
happened can look exactly like a real effect.

So the questions to keep asking are:

- **What is actually in this file?** Not what you assume &mdash; what is there.
- **Does this dataset support the question I am asking?**
- **How is the data being transformed?** Plot the data after each step.
- **What would make this result wrong?** Name it before you see the answer.

</div>

---

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Part 1: Load the dataset and find the pieces you need</h1>

Same access pattern as this morning: find your dataset's mount under <code>/data</code>, locate a
session's NWB file, then dot and bracket notation into the containers.

**Your classmate's figure tells you what to look for.** Before you open anything, list the pieces the
figure needs &mdash; neural activity, plus whatever else it plots: a behavioral trace, epoch
boundaries, trial times, stimulus identity.

Then find each one, and note the ones that turn out not to exist. **A piece being absent is a
finding about the dataset, not a failure.** Some datasets have no running wheel, no pupil camera, no
visual stimulus at all. You will build the figure from what is there.

</div>

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pynwb
from scipy import stats

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

data_dir = '/data'

In [ ]:
# List the datasets mounted under /data.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Start from the metadata table, not the file tree.</b> Each dataset has a metadata CSV in
<code>/code/metadata/</code> &mdash; one row per session, with subject, session type, date and the
asset name. Read that first and choose a session from it, because the filename alone will not tell you
which imaging stage or task condition you are looking at.

Then build the path: the NWB lives inside that dataset's mount under <code>/data/</code>.

</div>

In [ ]:
# EDIT: read your dataset's metadata CSV from /code/metadata and look at what it
# offers: how many subjects, how many session types, how many sessions each.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Which session does your classmate's figure come from? Use the table to find it
&mdash; subject, session type, date &mdash; and say what you filtered on.

Look at what the table offers before you filter. How many subjects, how many session types, how many
sessions each? That inventory is the first thing you know about the dataset.

</div>

In [ ]:
# EDIT: filter the table to the session behind your figure, take one row as
# `session`, and say what you filtered on.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Now build the path.</b> An NWB file is either a single <code>.nwb</code> file (HDF5) or a
<code>.nwb.zarr</code> <i>directory</i>, and datasets here are packaged by different groups &mdash; the
file may sit at the top of the mount or a few levels down. Search for it rather than hardcoding a
path, and check you got exactly one match.

</div>

In [ ]:
def find_nwb(root, contains=None, max_depth=3):
    """Return paths to .nwb files and .nwb.zarr directories under `root`.

    `contains` filters to paths containing that string -- pass the session id.
    """
    found = []
    root = root.rstrip('/')
    base_depth = root.count(os.sep)
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath.count(os.sep) - base_depth >= max_depth:
            dirnames[:] = []
        # A .nwb.zarr directory IS the file -- do not descend into its internals.
        for d in list(dirnames):
            if d.endswith('.nwb.zarr'):
                found.append(os.path.join(dirpath, d))
                dirnames.remove(d)
        for f in filenames:
            if f.endswith('.nwb'):
                found.append(os.path.join(dirpath, f))
    if contains is not None:
        found = [p for p in found if contains in p]
    return sorted(found)

In [ ]:
# EDIT: set `dataset_dir` to the mount holding your dataset (one of the names
# printed above), then use find_nwb() to locate the NWB file for your session.
# Check you got exactly one match before continuing.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Did you get exactly one match? More than one usually means several processing
generations of the same session are attached &mdash; check which you picked. Zero means the session
in the table is not mounted in this capsule, which is worth knowing before you debug anything else.

</div>

In [ ]:
# Open the NWB file. Name it `nwb`.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Find the data the figure needs</h3>

Four containers hold almost everything. Which one holds what **varies by dataset**, so list all four
before you index into any of them.

| container | commonly holds |
| --- | --- |
| `processing` | processed neural activity, and often behavior |
| `intervals` | epoch tables, trial tables, stimulus presentation tables |
| `stimulus` | stimulus templates &mdash; but in some datasets, the trial tables too |
| `acquisition` | raw acquired signals |

The last row is not hypothetical: some datasets put their trial tables in `stimulus` and leave
`intervals` holding only epochs. If you look in one container, find nothing, and conclude the data
is missing, you will be wrong. **Print all four.**

</div>

In [ ]:
# What is in this file? Print the containers before you index into any of them.
# processing, intervals, acquisition, stimulus -- and remember a container can
# hold things the top-level listing does not show.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Is your dataset continuous or spiking?</b> This is the first fork in the road, and it
changes what "activity" even means.

<b>Continuous</b> (calcium imaging, LFP): a `(n_timepoints, n_cells)` array already exists in the
file. Find it and you are done.

<b>Spiking</b> (Neuropixels, sorted electrophysiology): there is no such array. Each unit carries its
own list of spike times, usually in a `units` table, and you must <b>bin</b> them yourself &mdash;
choose a bin width, count spikes per bin, divide by the width to get a rate in spikes/s. Everything
downstream then works the same way.

Two decisions come with spiking data, and neither has a default:

- <b>Which units.</b> Spike sorting produces more units than you should analyze. There will be
  quality-control columns (`is_qc_pass`, `firing_rate`, `presence_ratio`, `snr`) and often an
  anatomical label. Select on them explicitly and say what you selected &mdash; a session can drop
  from thousands of units to dozens, and the ones you drop change your answer.
- <b>Bin width.</b> Too wide blurs the response; too narrow leaves mostly-empty bins and noisy
  single-trial estimates. Try a few and see how much your answer moves.

<pre>
bin_width = 0.010                                    # seconds -- your decision
edges = np.arange(0, t_end + bin_width, bin_width)
counts, _ = np.histogram(one_unit_spike_times, bins=edges)
rate = counts / bin_width                            # spikes/s
bin_centres = edges[:-1] + bin_width / 2
</pre>

Sparse binned spikes behave like a deconvolved calcium trace: sharper in time, and noisy per
trial.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Two things to check as you pull out the activity trace.</b>

<b>Timestamps.</b> Some datasets store an explicit `timestamps` array; others store a sampling
`rate` and a `starting_time`, and you reconstruct the times yourself. Everything downstream needs
real times in seconds, so check which you have &mdash; `series.timestamps` is `None` when the file
uses a rate.

<b>Lazy loading.</b> NWB data objects do not load until you index them. That is what lets you open a
50&nbsp;GB file instantly, but it means `data.std()` may fail where `np.std(data)` works. Convert
with `np.asarray()` once you know the array is small enough to hold, or slice first.

</div>

In [ ]:
# Find the neural activity in this file and pull out two things:
#   `dff` -- the (n_timepoints, n_cells) trace array
#   `ts`  -- the matching times in seconds
# Not every dataset stores a timestamps array; see the note above.
# Print the shapes, the frame rate, and the session duration as a sanity check.

In [ ]:
# Pull out the other pieces your chosen figure needs -- stimulus/trial tables,
# and any behavioral traces the dataset has. Tables become DataFrames with
# .to_dataframe(); timeseries have .data and .timestamps.
# Print the shape of each, and note anything that turns out not to exist.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Quality control: which cells or units belong in the analysis?</h3>

Segmentation and spike sorting are automated, and both over-produce. An ophys plane contains ROIs the
classifier thinks are not cell bodies; a sorted probe contains units that drift, that are barely
above noise, or that are two neurons merged. <b>The activity matrix you just loaded usually contains
all of them.</b>

Pipelines record their own verdicts. For imaging they live on the ROI table beside the masks; for
electrophysiology, on the units table. The columns differ by pipeline and by dataset &mdash; boolean
flags, continuous probabilities, morphology metrics, contamination estimates &mdash; so there is no
list to memorise. Print the columns and see what your dataset offers.

Filtering is not automatically the right move, and the criteria are yours to justify. But
<b>inheriting the unfiltered set by default is a decision you made without noticing</b>, and it is the
kind that never appears in a methods section.

</div>

In [ ]:
# EDIT: find the per-cell quality table for your dataset -- a plane segmentation
# for imaging, nwb.units for electrophysiology -- and print its scalar columns:
# which are flags, which are continuous scores, and what each would exclude.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Does your dataset carry per-cell or per-unit quality metrics? Report what the
columns are, how many entries each flag would exclude, and whether the activity matrix is already
filtered or contains everything.

Then decide. Whatever you choose, **state the criterion and the count you dropped** &mdash; that
sentence belongs in your methods.

</div>

In [ ]:
# EDIT: apply your QC criterion. Check the table length matches the activity
# matrix first, apply the SAME mask to every per-cell array you loaded, and print
# how many cells you dropped.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Plotting a long recording.</b> A whole session at a fine sampling rate can be hundreds of
thousands of points &mdash; slow to draw and impossible to read. Plot a slice instead, but choose the
slice from the data rather than picking a round number: an arbitrary window can easily contain no
activity at all, and an empty panel looks identical to a broken one.

</div>

In [ ]:
# Plot one cell's trace against time, as a sanity check on what you loaded.
# Choose the cell deliberately rather than taking index 0, and say how you chose.
# Watch out for cells that are entirely NaN.
# If the recording is long, plot a slice -- and pick the slice from the data.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Look at that trace for a few seconds before moving on. Is anything about it
surprising? Would you have noticed if you had skipped straight to the analysis?

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The variables the rest of the notebook uses</h3>

**This is the only cell you edit to run this notebook on a different dataset.** Point these names at
the equivalent pieces of your NWB file; nothing after this cell refers to a dataset by name.

</div>

In [ ]:
# =====================================================================
# FILL IN FOR YOUR DATASET
# Everything after this cell uses these five names. Point them at the
# equivalent pieces of your NWB file.
# =====================================================================
activity = ...              # (n_timepoints, n_cells)
timestamps = ...            # (n_timepoints,) in seconds
events = ...                # one row per event / trial / presentation

CONDITION_COLUMN = ...      # column that labels the condition
ONSET_COLUMN = ...          # column that gives the event time
# =====================================================================

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Print the columns of your stimulus table. Which describe *what was
presented*, which describe *what the animal did*, and which are bookkeeping?

Note any column whose meaning you cannot guess &mdash; that is a databook lookup for your README.

</div>

In [ ]:
# Print the columns of your event table, then look at the first few rows.

---

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Part 2: Reproduce the figure, and interrogate what it shows</h1>

You have your classmate's figure. You do not have their code, and you may not have a caption either.

<b>Before you write anything, write down what you think the figure shows.</b> One or two sentences,
in your notebook, as a claim someone could disagree with: <i>"activity is higher during X than during
Y"</i>, <i>"the response is larger on this trial type"</i>, <i>"these two signals rise together."</i>

Two reasons this comes first. It commits you to an interpretation before the data can talk you into
one &mdash; and it converts a picture into something you can actually test. A figure cannot be right
or wrong. A claim can.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Write your claim about the figure you picked, in the cell below, before you
write any code.

Be specific enough to be wrong. "There is neural activity" is not a claim; "population activity is
higher in the second half of the session" is.

</div>

_Your claim:_

<!-- write it here -->


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Is your timeseries actually continuous?</b>

Most of the time it is. Many datasets record straight through, and this check takes one line and
returns nothing interesting. Do it anyway, because when it <i>does</i> find something the default
plot will not tell you.

Some acquisition systems write behavioral streams per block rather than continuously, so an
experimenter break, a rig adjustment or a script restart can leave an interval with no samples at
all &mdash; the timestamps just jump. Whether that happened is a property of how your session was
run, not something you can infer from the plot.

And the plot will not warn you: <code>plot()</code> draws a straight line between consecutive samples
regardless of how far apart in time they are, so a ten-minute hole becomes a flat segment that looks
exactly like an animal sitting still. <b>Missing data and zero are not the same thing, and the
default plot renders them identically.</b>

So: check the interval between samples once, before you trust any flat stretch. If there are no
gaps &mdash; the common case &mdash; you have spent one cell and can plot normally. If there are, use
the helper below to break the line at them instead of interpolating across.

</div>

In [ ]:
def break_gaps(t, y, factor=10):
    """Insert NaN wherever sampling pauses, so plots don't interpolate across it.

    A gap is any interval more than `factor` times the typical one.
    """
    dt = np.diff(t)
    typical = np.median(dt)
    breaks = np.flatnonzero(dt > factor * typical)
    if len(breaks) == 0:
        return t, y
    return (np.insert(np.asarray(t, float), breaks + 1, np.nan),
            np.insert(np.asarray(y, float), breaks + 1, np.nan))

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** For each timeseries in your figure, report the typical sampling interval and
whether any interval is much larger than it.

**Finding none is the likely outcome and a perfectly good answer** &mdash; write down that you
checked and the trace is continuous. If you do find gaps, note how much of the session they cover and
which streams are affected; they may not be the same for every signal.

</div>

In [ ]:
# For each timeseries your figure uses: the typical sampling interval, how many
# gaps are much larger than it, and how much of the session has no data.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Now rebuild it</h3>

Get the pieces the figure needs and plot them. You will not match it exactly &mdash; different
smoothing, different colors, a different subset of cells &mdash; and that is fine. What matters is
that the structure you see is the same structure they saw.

If you cannot rebuild some element because the dataset does not contain it, note that and rebuild
what you can.

</div>

In [ ]:
# Compute whatever your chosen figure shows.
# For a population average: the mean across cells at each timepoint.
# Name it `population`, and check its shape before you plot.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

To shade the epochs we need their start and stop times. Where epochs live varies by dataset:
sometimes an `epoch_name` column on the stimulus table, sometimes a separate epochs table.

**Check that the column you group by actually varies.** If it takes one value, you will get a single
block spanning the session &mdash; a figure that looks fine and is wrong.

</div>

In [ ]:
# Does your event table carry an epoch column? If so, how many distinct
# values does it actually take?

In [ ]:
# Build a table of epoch boundaries with one row per epoch, indexed by name
# and sorted in time. Name it `epochs`, with `start_time` and `stop_time`
# columns -- the shading helper below expects those.

In [ ]:
# Shade each epoch a different color -- same helper as the tutorial
colors = dict(zip(epochs.index, plt.cm.Pastel1.colors))


def shade_epochs(ax):
    """Shade each epoch on ax, one color per epoch label."""
    for label, row in epochs.iterrows():
        ax.axvspan(row.start_time, row.stop_time, color=colors[label], alpha=0.5, zorder=0,
                   label=label)

In [ ]:
# Build your version of the figure: the traces stacked on a shared time axis,
# with the epochs shaded (use `shade_epochs`). Include only the streams your
# dataset actually has.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Now test the claim you wrote above &mdash; do not eyeball it.

Turn your sentence into a number you can check. If it compares epochs, compute the mean in each one,
alongside how long each epoch lasted, when in the session it happened, and what the animal was doing.
If it compares something else, compute the equivalent.

Before you look: **what would make this comparison unfair?** Write your answer down first, then see
whether the table bears it out.

Then go back and mark your claim as supported, contradicted, or untestable with this data. All three
are legitimate outcomes and all three belong in your README.

</div>

In [ ]:
# For each epoch compute the mean activity, and alongside it the things that
# could confound the comparison: how long the epoch lasted, how many samples
# that is, when in the session it happened, and what the animal was doing.
# Build a DataFrame with one row per epoch.

---

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Part 3: Align activity to event onsets</h1>

The session overview shows everything at once, which means it shows very little. To see a response
you need to **align** activity to the times when something happened, and look across repeats.

"Something happened" need not be a visual stimulus. It might be a sound, an optogenetic pulse, a
reward, a lick, or the start of a trial. Anything with a repeatable onset time works the same way
&mdash; and the rest of this notebook says "event" rather than "stimulus" for that reason.

This morning's tutorial averaged across presentations. Here we look at what the average hides.

</div>

In [ ]:
# How many times was each condition presented?

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Are all of these events the same kind of event?</h3>

An event table usually contains rows that are **not equivalent trials**. Depending on the dataset
that might be first versus repeated presentations, rewarded versus unrewarded trials, different
stimulus families, trials the animal responded to versus ignored, blocks recorded before and after
a manipulation, or blank and omitted entries that are not events at all.

This matters before you align anything, for two reasons:

- **Response magnitude can differ several-fold between trial types.** Averaging them together dilutes
  the response toward whichever type is most numerous &mdash; which is often the weakest one.
- **Trial types differ in what else is happening.** Reward, licking, and arousal ride along with some
  trial types and not others, so a difference you attribute to the stimulus may not be about the
  stimulus.

Find the columns in your table that distinguish trial types, and count them.

</div>

In [ ]:
# Which columns in your event table distinguish different KINDS of trial?
# Find them and count the rows of each kind.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

To compare them we need to cut a window of data around each onset. Same
`align_to_events` helper as this morning's tutorial.

</div>

In [ ]:
def align_to_events(data, timestamps, event_times, pre=0.5, post=1.5):
    """Cut a window of data around each event time.

    Returns (windows, t) where t is time in seconds relative to the event.
    """
    dt = np.median(np.diff(timestamps))
    n_pre, n_post = int(pre / dt), int(post / dt)

    windows = []
    for event_time in event_times:
        i = np.searchsorted(timestamps, event_time)

        # skip events too close to the start or end of the recording
        if i - n_pre >= 0 and i + n_post <= len(timestamps):
            windows.append(data[i - n_pre:i + n_post])

    t = np.arange(-n_pre, n_post) * dt
    return np.array(windows), t

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Pick two trial types from your table and align the population average to
each separately, then plot both on the same axes.

Write down your prediction first: do you expect a difference, and how large?

Then choose which type to carry forward, and one condition within it. Name three things, because the
rest of Part 3 refers to them:

| name | what it holds |
| --- | --- |
| `onsets_pool` | onset times of ALL trials of your chosen type |
| `onsets` | onset times of the one condition you picked |
| `ROI` | index of your example cell |

</div>

In [ ]:
# Pick two kinds of trial and compare them: align the population average to
# each set of onsets separately and plot both on the same axes.
# Subtract each window's own pre-onset baseline before averaging.
# Name the two onset arrays `group_a` and `group_b`.

In [ ]:
# Choose the trial type to carry forward, and one condition within it.
# Name them:
#   `onsets_pool` -- onsets of ALL trials of that type
#   `onsets`      -- onsets of the one condition you picked

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Which cell or unit to look at?</h3>

Whatever your dataset calls them &mdash; ROIs in an imaging plane, sorted units on a probe &mdash;
taking the first one in the table is an arbitrary choice you did not disclose. Ranking by how strongly
they respond is a *different* undisclosed choice unless you say so. Pick deliberately and write down
how you picked.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>Which signal do you align?</b> Most datasets ship more than one representation of the
same activity, and the choice is yours &mdash; but it is a choice, and it changes what the figures
show.

<table>
<tr><td><b>&Delta;F/F</b> (imaging)</td><td>Continuous fluorescence. Carries the indicator's rise and
decay, so a brief response is smeared forward by hundreds of milliseconds, and slow drift shared
across the field of view inflates correlations between any two cells. Every timepoint has a
value.</td></tr>
<tr><td><b>Deconvolved events</b> (imaging)</td><td>An estimate of when the cell actually fired, with
the indicator kinetics removed. Temporally tighter, and mostly exact zeros &mdash; so single-trial
estimates are much noisier even though the trial average looks cleaner.</td></tr>
<tr><td><b>Spike times</b> (electrophysiology)</td><td>Discrete times, no continuous trace at all. You
choose a bin width to get a matrix, and that width is a real analysis decision: too fine and every
bin is empty, too coarse and you lose the timing you came for.</td></tr>
</table>

None of these is the correct one. A question about response <i>latency</i> or duration is badly served
by &Delta;F/F; a question needing a reliable per-trial number is badly served by a sparse signal. Pick
one, say why, and if you have time run the analysis twice and compare &mdash; that comparison is
usually more informative than either result alone.

<b>Set the choice in one place</b> so switching it is a one-line edit rather than a rewrite.

</div>

In [ ]:
# EDIT: which signal to align for the figures below -- the deconvolved events if
# your dataset has them, otherwise the continuous trace. Set `aligned_signal` and
# a `SIGNAL_LABEL` string for the axis labels, and say why you chose it.

In [ ]:
# Choose an example cell to look at, and say how you chose it.
# Set `pre` and `post` (seconds before/after onset) and name the cell `ROI`.
# Re-derive the index against the CURRENT activity matrix -- an index from an
# earlier cell may not refer to the same neuron.

In [ ]:
# Cut a window around every onset for your example cell, using
# `align_to_events`. Name the results `windows` and `t`, and print the shape.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Compare the number of windows you got back against the number of onsets you
asked for. Are they the same?

If not, read the helper again and work out where the missing trials went &mdash; then decide whether
losing them matters for your analysis.

This is worth doing every time you call something that returns one row per trial. A function that
quietly returns fewer rows than you gave it will not raise an error; it will just make your
n smaller than you think it is.

</div>

In [ ]:
# Compare the number of onsets you asked for against the number of windows
# that came back.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Raster and PSTH</h3>

The raster shows every trial; the PSTH is their average. Plot them together so you can see what the
average discards.

</div>

In [ ]:
# Plot the raster and the PSTH side by side: every trial as a heatmap, and
# the trial average with a measure of spread.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Now do it for every cell and plot the result as a heatmap, sorted by
response magnitude. How many cells respond at all?

</div>

In [ ]:
# Do it for every cell: build a (n_cells, n_timepoints) array of
# trial-averaged responses. Name it `responses`.

In [ ]:
# Plot `responses` as a heatmap, sorted by response magnitude.
# Then plot it again with each cell's own pre-onset baseline subtracted,
# and compare the two panels.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>The same analysis on a different signal</h3>

<b>Skip this section if your dataset has only one representation of activity.</b> A probe recording
gives you spike times and nothing else &mdash; there is no second signal to compare against, and
saying so in your write-up is the correct answer here, not a gap.

If you do have two &mdash; a continuous trace and a deconvolved estimate, most commonly &mdash; they
are not interchangeable, and running the same analysis on both is the cheapest way to find out how
much your conclusion depends on that choice.

<b>Check what your dataset has before assuming.</b> List the interfaces in the processing container
and see whether a second per-cell timeseries is there at all.

</div>

In [ ]:
# Clean tiny float noise to exact zeros so "fraction exactly zero" means what
# it says, then compare the two representations: shapes, sparsity, shared clock.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** If your dataset has two activity representations, align both to the same
onsets and plot the trial-averaged population response side by side. What differs &mdash; the
duration, the shape, the size relative to baseline?

If it has only one, note that in your README and move on.

</div>

In [ ]:
# Build a list of the activity representations you have, then plot the aligned
# population response for each. If you only have one, say so and move on.

---

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Part 4: Signal and noise correlations</h1>

<h3>First, the math</h3>

The Pearson correlation between two variables $x$ and $y$ is

$$ r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}
              {\sqrt{\sum_i (x_i - \bar{x})^2}\;\sqrt{\sum_i (y_i - \bar{y})^2}} $$

In words:

1. **Center** each variable by subtracting its mean.
2. **Multiply** the centered values pointwise and sum &mdash; large and positive when they vary
   together, negative when oppositely, near zero when unrelated.
3. **Normalize** by each variable's spread, forcing the result between -1 and +1.

Compute it once by hand before running it thousands of times.

</div>

In [ ]:
# Compute the correlation between two cells' traces BY HAND, in three steps:
#   1. centre each variable (subtract its mean)
#   2. multiply the centred values pointwise and sum
#   3. normalise by the spread of each
# Then check your answer against np.corrcoef.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

Two consequences that matter for everything below:

- $r$ says nothing about response **size**, only whether two things move together.
- $r$ is computed over a set of paired observations, and **how many observations you have determines
  how noisy $r$ is** &mdash; but the value itself gives you no clue how many there were.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** What does a given value of $r$ look like? Simulate pairs with known
correlations and plot them.

</div>

In [ ]:
# Simulate pairs of variables with known correlations (try 0, 0.2, 0.5, 0.9)
# and plot each as a scatter, titled with its measured r.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Two reasons neurons are correlated</h3>

- **Signal correlation.** Do they respond similarly *across conditions*? Correlate the two neurons'
  tuning curves &mdash; their average response to each condition.
- **Noise correlation.** When the *same* condition repeats, do they fluctuate together around their
  own averages? Subtract each condition's mean and correlate the residuals.

A "condition" is whatever your event table repeats: an image, a grating direction, a tone, a
photostimulation target, a task context. All that matters is that it recurs enough times to average
over.

Same data, different thing averaged over:

| | what is correlated | one observation is |
| --- | --- | --- |
| signal | condition means | one condition |
| noise | within-condition residuals | one trial |

That last column matters more than anything else in this notebook.

</div>

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 1: choose which events to use</h3>

Not every event is comparable to every other. Decide which subset is a fair comparison and write down
why.

</div>

In [ ]:
# Restrict to comparable events, and write down WHY -- this choice belongs in
# your methods. Name the results:
#   `onsets_all` -- the onset times you keep
#   `labels`     -- the condition label for each of those onsets

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 2: one number per trial per neuron</h3>

We need a `(n_trials, n_cells)` matrix. Average each aligned window over a response window, and
subtract a **baseline** from just before onset &mdash; otherwise each trial's "response" includes
wherever the cell happened to be sitting beforehand, and those levels drift together across the
population from bleaching, arousal, and movement.

<b>Choosing the two windows is dataset-specific.</b> The response window should cover the response
your Part 3 plot showed &mdash; look at it rather than copying a number from here, since a calcium
signal and a spike rate need very different windows. The baseline window should sit in the gap
before onset, and must **exclude any stimulation artifact**: with optogenetics or electrical
stimulation the frames around the pulse can be unusable, so leave a margin on both sides.

</div>

In [ ]:
# Build the (n_trials, n_cells) response matrix `R`: for each trial, the mean
# activity in a response window minus a pre-onset baseline.
# Choose both windows deliberately -- see the note above.
# Keep `labels_used` in register with the rows of `R`.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 3: tuning curves &mdash; look before correlating</h3>

</div>

In [ ]:
# Build the tuning curves: the per-condition mean response of each cell.
# Name the condition list `conditions` and the array `tuning`,
# shaped (n_conditions, n_cells).

In [ ]:
# Plot the tuning curves before correlating anything: a few cells as lines,
# and all cells as a heatmap.

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** How many numbers make up one neuron's tuning curve?

That is how many paired observations each signal correlation gets. Write it down.

</div>

In [ ]:
# How many numbers make up one tuning curve, and how many trials are available
# for the noise correlations? Print both.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 4: residuals &mdash; look before correlating</h3>

Subtract **each condition's own mean**, not the grand mean. Subtracting the grand mean would leave
the differences between conditions in the residuals, making your "noise" correlation partly a signal
correlation.

</div>

In [ ]:
# Build the residuals: subtract each condition's OWN mean from its trials.
# Name the array `residuals`, and check that its overall mean is ~0.

In [ ]:
# Plot the raw responses and the residuals for one cell, side by side, so you
# can see what subtracting the condition means removed.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Step 5: correlate</h3>

`np.corrcoef` correlates **rows**, so transpose to get cells rather than trials. Getting this
backwards produces a plausible matrix of entirely the wrong thing &mdash; check the output shape
against the number of cells.

</div>

In [ ]:
# Compute the two correlation matrices, `C_signal` and `C_noise`.
# Watch the orientation: np.corrcoef correlates ROWS.
# Take each pair once with np.triu_indices -- name the index `pairs`, and the
# extracted values `signal_values` and `noise_values`.
# Print the mean of each, with how many observations went into it.

In [ ]:
# Plot the two matrices side by side, plus signal against noise correlation
# for every pair. Scale each matrix to its own range so neither saturates.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Is this result trustworthy?</h1>

Every number so far is a point estimate with no error bar. The single most useful check: **would you
get the same answer with half the data?**

Split trials in half at random, compute the correlations on each half separately, and correlate the
two halves' answers. Split **within each condition** so both halves see every condition.

Three outcomes, and all three are informative:

- **One high, one low** &mdash; trust the high one, and say why the other is not trustworthy.
- **Both high** &mdash; you have enough data for both; proceed.
- **Both near zero** &mdash; report that. It usually means the condition variable you chose does not
  organise these neurons' responses, however well-balanced it looked in the inventory. That is a
  real result about your dataset, and it is a better README than a matrix you cannot defend.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Before running it &mdash; which do you expect to be more reliable, signal or
noise correlations? Look back at the observation counts you wrote down.

</div>

In [ ]:
def correlations_from(R_sub, labels_sub):
    """Signal and noise correlation matrices from a subset of trials.

    Wrap up what you did above so it can be re-run on any subset of trials:
      - tuning    = per-condition means             -> signal correlation
      - residuals = each condition's mean removed   -> noise correlation

    Returns (C_signal, C_noise).
    """
    # your code here

In [ ]:
# Split-half reliability. Split the trials in half WITHIN each condition,
# compute the correlation matrices on each half with `correlations_from`, and
# correlate the two halves' answers (scipy.stats.spearmanr on `pairs`).
# Repeat ~10 times; report the mean and spread for signal and for noise.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h3>Does the signal you chose change the answer?</h3>

Everything so far used one representation of activity. If your dataset provides a second one, repeat
the whole chain on it and compare the numbers that matter. If it provides only one, note that and
move on.

To repeat the chain you need the response-matrix construction as a reusable function rather than a
one-off block &mdash; so wrap it, the same way you wrapped the correlations.

</div>

In [ ]:
def response_matrix(A):
    """Build the (n_trials, n_cells) response matrix for a given signal.

    Same steps you used to build `R`, wrapped so it can be applied to either
    activity representation.
    """
    # your code here

In [ ]:
# Run the whole chain on each activity representation your dataset has, and
# compare: mean signal and noise correlation, their split-half reliabilities,
# and what fraction of the single-trial responses are exactly zero.

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<b>One column may not be the whole condition.</b>

A column can look like a clean condition variable &mdash; many levels, perfectly balanced &mdash;
while the stimulus varied in some <i>other</i> way at the same time. Two trials sharing that column's
value are then not repeats of the same thing, and averaging them together destroys the tuning you
were trying to measure.

Receptive-field mapping is the classic case: orientation is balanced, but the stimulus also moves
around the screen, so "144 repeats of 45&deg;" is really a handful of repeats at each of many
positions. The same trap appears whenever a design crosses two factors and you only notice one.

Check for it by asking what else varies across the trials you just called identical. Group by your
condition column, look at the other columns within a group, and see whether they are constant. If
they are not, either restrict to one level of the other factor, or make the condition the
<i>combination</i> of both.

</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">

**Exercise:** Signal correlations need a condition that repeats. Does your dataset have
one?

Inventory the candidate columns: how many distinct values, how many repeats, how balanced.

Then answer **two separate questions**, because they can disagree:

1. **Is the analysis possible?** Does some column have enough conditions with enough repeats?
2. **Is it meaningful?** Does that column label something you would expect neurons to be tuned
   *to*, in a way that a correlation across condition means would capture?

A column can pass the first test and fail the second. State a verdict on both, and check it against
your reliability numbers.

</div>

In [ ]:
# Inventory the candidate condition columns in your event table: for each,
# how many distinct values, the repeats of the least and most common, and how
# balanced. Then state your verdict on both questions above.

---

<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

<h1>Summary</h1>

<h3>The process</h3>

1. **Find out what is in the file** before analyzing it &mdash; and check that the dataset supports
   your question. Sometimes the answer is no.
2. **Plot the data after each transformation.** Single trials before averages; tuning curves before
   correlations.
3. **Name every decision.** Event subset, condition column, response window, baseline. Each is a
   fork, and each belongs in your methods.
4. **Try to break your own result.** Split the data in half and see if the answer survives.
5. **Let the dataset answer back.** If the check says your result is noise, or the dataset has no
   variable that supports your question, that is the finding. Report it rather than reaching for the
   analysis you planned to run.

<h3>Traps this notebook demonstrated</h3>

| trap | how you catch it |
| --- | --- |
| A result from few observations looks like one from many | split-half reliability |
| A well-balanced condition variable that means nothing | reliability, not the inventory |
| A condition column that hides a second varying factor | group by it, check what else moves |
| Analyzing units that should have been dropped | select on quality columns, and say so |
| A helper function silently drops data | compare output shape to input |
| A column exists but carries no information | check that it actually varies |
| One bad trial turns every cell's score into NaN | count your NaNs; use `nanmean` |
| Epoch comparisons confounded with time and behavior | check durations, order, behavior |
| An example cell chosen to look good | state your selection rule |
| Data looks absent but is stored elsewhere | look in every container first |
| An index from an earlier cell after reshaping the data | re-derive indices, never carry them |

<h3>Why this matters</h3>

You can generate an analysis faster than you can validate one. The only defense is to know your data
well enough that a wrong answer looks wrong to **you** &mdash; because it will not look wrong to the
code, and it will not look wrong on the plot.

</div>